In [1]:
import numpy as np
import os
import torch
import sys
sys.path.append("/data/nishome/user1/chaochuan/TSGym_benchmark")
from models.TSGym import Model as TSGym
print(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

/data/nishome/user1/miniconda3/envs/mqenv/lib/python3.11/site-packages/local_attention/rotary.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/data/nishome/user1/miniconda3/envs/mqenv/lib/python3.11/site-packages/local_attention/rotary.py:55: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)


cuda


In [ ]:
# logger.py


DB_PATH = "test.db"




In [6]:
init_db()


In [ ]:
# gpu_monitor.py
import pynvml
import threading
import time
import os

pynvml.nvmlInit()

class GPUMemoryMonitor:
    def __init__(self, interval=0.5):
        self.interval = interval
        self.max_mem = 0
        self.running = False
        self.handle = None

        pid = os.getpid()
        device_count = pynvml.nvmlDeviceGetCount()

        # 找到当前进程所在 GPU
        for i in range(device_count):
            h = pynvml.nvmlDeviceGetHandleByIndex(i)
            infos = pynvml.nvmlDeviceGetComputeRunningProcesses(h)
            for p in infos:
                if p.pid == pid:
                    self.handle = h
                    break
            if self.handle:
                break

    def start(self):
        if not self.handle:
            print("No GPU process found.")
            return

        self.running = True
        def run():
            while self.running:
                infos = pynvml.nvmlDeviceGetComputeRunningProcesses(self.handle)
                for p in infos:
                    if p.pid == os.getpid():
                        self.max_mem = max(self.max_mem, p.usedGpuMemory / 1024**2)
                time.sleep(self.interval)

        self.thread = threading.Thread(target=run)
        self.thread.start()

    def stop(self):
        self.running = False
        if hasattr(self, "thread"):
            self.thread.join()
        return self.max_mem


In [10]:
def run_fn():
    device = torch.device("cuda")
    a = torch.random.rand(100,2)
    a.to(device)
    sleep(10)
    return "max"

In [11]:
def run_experiment(exp_setting, run_fn):
    log_start(exp_setting)

    monitor = GPUMemoryMonitor()
    monitor.start()

    try:
        result = run_fn()  # 你的实验函数
        max_mem = monitor.stop()
        log_end(exp_setting, result_metric=result, max_gpu_mem=max_mem)

    except Exception as e:
        max_mem = monitor.stop()
        log_end(exp_setting, result_metric=None, max_gpu_mem=max_mem, error_msg=str(e))
        raise e  # 保持原始异常行为


In [2]:
import sqlite3
import pandas as pd

DB_PATH = "../liangshaung_v100.db"

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("SELECT * FROM exp_logs", conn)
conn.close()

df


,exp_setting,start_time,end_time,duration_sec,status,max_gpu_mem_MB,result_metric,error_msg
0,LTF_TSGym13104_True_False_None_None_False_seri...,2025-12-21 16:20:09,2025-12-21 16:20:09,0.0,FAILED,NaN,null,CUDA error: CUDA-capable device(s) is/are busy...
1,LTF_TSGym110530_False_True_RevIN_DFT_True_seri...,2025-12-21 16:20:09,2025-12-21 16:20:09,0.0,FAILED,NaN,null,CUDA error: CUDA-capable device(s) is/are busy...
2,LTF_TSGym110710_False_True_RevIN_MoEMA_False_s...,2025-12-21 16:20:09,2025-12-21 16:20:09,0.0,FAILED,NaN,null,CUDA error: CUDA-capable device(s) is/are busy...
3,LTF_TSGym14184_False_False_RevIN_MoEMA_False_s...,2025-12-21 16:20:09,2025-12-21 16:20:09,0.0,FAILED,NaN,null,CUDA error: CUDA-capable device(s) is/are busy...
4,LTF_TSGym14184_False_False_RevIN_MoEMA_False_s...,2025-12-21 16:20:17,2025-12-21 16:20:18,1.0,FAILED,NaN,null,CUDA error: CUDA-capable device(s) is/are busy...
...,...,...,...,...,...,...,...,...
4467,LTF_TSGym13304_False_True_Stat_DFT_False_serie...,2025-12-28 18:35:26,None,NaN,RUNNING,NaN,None,None
4468,LTF_TSGym10673_True_False_None_MoEMA_True_seri...,2025-12-28 18:59:34,2025-12-28 19:37:54,2300.0,FINISHED,0.0,"""file:ETTh1.csv, mse:0.46410152316093445, mae:...",None
4469,LTF_TSGym12592_False_False_RevIN_DFT_True_seri...,2025-12-28 19:23:35,None,NaN,RUNNING,NaN,None,None
4470,LTF_TSGym10673_True_False_None_MoEMA_True_seri...,2025-12-28 19:38:04,2025-12-28 20:10:52,1968.0,FINISHED,0.0,"""file:ETTh1.csv, mse:0.48394089937210083, mae:...",None


In [11]:
interval = 1
max_mem = 0
running = False
handle = None

pid = os.getpid()
device_count = pynvml.nvmlDeviceGetCount()

# 找到当前进程所在 GPU
for i in range(device_count):
    h = pynvml.nvmlDeviceGetHandleByIndex(i)
    infos = pynvml.nvmlDeviceGetComputeRunningProcesses(h)
    for p in infos:
        if p.pid == pid:
            handle = h
            break
    if handle:
        break

In [12]:
handle

In [13]:
device_count

3

In [16]:
pynvml.nvmlDeviceGetComputeRunningProcesses(pynvml.nvmlDeviceGetHandleByIndex(0))[0].pid

812376

In [17]:
pid

1204456

In [2]:
import sys
import argparse

# 手动清除 Jupyter 自动传递的参数
sys.argv = [arg for arg in sys.argv if not arg.startswith('--f=')]

parser = argparse.ArgumentParser(description='TimesNet')

# basic config
parser.add_argument('--task_name', type=str, required=False, default='long_term_forecast',
                    help='task name, options:[long_term_forecast, short_term_forecast, imputation, classification, anomaly_detection]')
parser.add_argument('--is_training', type=int, required=False, default=1, help='status')
parser.add_argument('--model_id', type=str, required=False, default='test', help='model id')
parser.add_argument('--model', type=str, required=False, default='Autoformer',
                    help='model name, options: [Autoformer, Transformer, TimesNet]')

# data loader
parser.add_argument('--data', type=str, required=False, default='ETTm1', help='dataset type')
parser.add_argument('--root_path', type=str, default='./data/ETT/', help='root path of the data file')
parser.add_argument('--data_path', type=str, default='ETTh1.csv', help='data file')
parser.add_argument('--features', type=str, default='M',
                    help='forecasting task, options:[M, S, MS]; M:multivariate predict multivariate, S:univariate predict univariate, MS:multivariate predict univariate')
parser.add_argument('--target', type=str, default='OT', help='target feature in S or MS task')
parser.add_argument('--freq', type=str, default='h',
                    help='freq for time features encoding, options:[s:secondly, t:minutely, h:hourly, d:daily, b:business days, w:weekly, m:monthly], you can also use more detailed freq like 15min or 3h')
parser.add_argument('--checkpoints', type=str, default='./checkpoints', help='location of model checkpoints')

# forecasting task
parser.add_argument('--seq_len', type=int, default=96, help='input sequence length')
parser.add_argument('--label_len', type=int, default=48, help='start token length')
parser.add_argument('--pred_len', type=int, default=96, help='prediction sequence length')
parser.add_argument('--seasonal_patterns', type=str, default='Monthly', help='subset for M4')
parser.add_argument('--inverse', action='store_true', help='inverse output data', default=False)

# inputation task
parser.add_argument('--mask_rate', type=float, default=0.25, help='mask ratio')

# anomaly detection task
parser.add_argument('--anomaly_ratio', type=float, default=0.25, help='prior anomaly ratio (%)')

# model define
parser.add_argument('--expand', type=int, default=2, help='expansion factor for Mamba')
parser.add_argument('--d_conv', type=int, default=4, help='conv kernel size for Mamba')
parser.add_argument('--top_k', type=int, default=5, help='for TimesBlock')
parser.add_argument('--num_kernels', type=int, default=6, help='for Inception')
parser.add_argument('--enc_in', type=int, default=7, help='encoder input size')
parser.add_argument('--dec_in', type=int, default=7, help='decoder input size')
parser.add_argument('--c_out', type=int, default=7, help='output size')
parser.add_argument('--d_model', type=int, default=512, help='dimension of model')
parser.add_argument('--n_heads', type=int, default=8, help='num of heads')
parser.add_argument('--e_layers', type=int, default=2, help='num of encoder layers')
parser.add_argument('--d_layers', type=int, default=1, help='num of decoder layers')
parser.add_argument('--d_ff', type=int, default=2048, help='dimension of fcn')
parser.add_argument('--moving_avg', type=int, default=25, help='window size of moving average')
parser.add_argument('--factor', type=int, default=1, help='attn factor')
parser.add_argument('--distil', action='store_false',
                    help='whether to use distilling in encoder, using this argument means not using distilling',
                    default=True)
parser.add_argument('--dropout', type=float, default=0.1, help='dropout')
parser.add_argument('--embed', type=str, default='timeF',
                    help='time features encoding, options:[timeF, fixed, learned]')
parser.add_argument('--activation', type=str, default='gelu', help='activation')
parser.add_argument('--channel_independence', type=int, default=1,
                    help='0: channel dependence 1: channel independence for FreTS model')
parser.add_argument('--decomp_method', type=str, default='moving_avg',
                    help='method of series decompsition, only support moving_avg or dft_decomp')
parser.add_argument('--use_norm', type=int, default=1, help='whether to use normalize; True 1 False 0')
parser.add_argument('--down_sampling_layers', type=int, default=0, help='num of down sampling layers')
parser.add_argument('--down_sampling_window', type=int, default=1, help='down sampling window size')
parser.add_argument('--down_sampling_method', type=str, default=None,
                    help='down sampling method, only support avg, max, conv')
parser.add_argument('--seg_len', type=int, default=48,
                    help='the length of segmen-wise iteration of SegRNN')

# optimization
parser.add_argument('--num_workers', type=int, default=10, help='data loader num workers')
parser.add_argument('--itr', type=int, default=1, help='experiments times')
parser.add_argument('--train_epochs', type=int, default=10, help='train epochs')
parser.add_argument('--batch_size', type=int, default=32, help='batch size of train input data')
parser.add_argument('--patience', type=int, default=3, help='early stopping patience')
parser.add_argument('--learning_rate', type=float, default=0.0001, help='optimizer learning rate')
parser.add_argument('--des', type=str, default='test', help='exp description')
parser.add_argument('--loss', type=str, default='MSE', help='loss function')
parser.add_argument('--lradj', type=str, default='type1', help='adjust learning rate')
parser.add_argument('--use_amp', action='store_true', help='use automatic mixed precision training', default=False)

# GPU
parser.add_argument('--use_gpu', type=bool, default=True, help='use gpu')
parser.add_argument('--gpu', type=int, default=0, help='gpu')
parser.add_argument('--use_multi_gpu', action='store_true', help='use multiple gpus', default=False)
parser.add_argument('--devices', type=str, default='0,1,2', help='device ids of multile gpus')

# de-stationary projector params
parser.add_argument('--p_hidden_dims', type=int, nargs='+', default=[128, 128],
                    help='hidden layer dimensions of projector (List)')
parser.add_argument('--p_hidden_layers', type=int, default=2, help='number of hidden layers in projector')

# metrics (dtw)
parser.add_argument('--use_dtw', type=bool, default=False, 
                    help='the controller of using dtw metric (dtw is time consuming, not suggested unless necessary)')

# Augmentation
parser.add_argument('--augmentation_ratio', type=int, default=0, help="How many times to augment")
parser.add_argument('--seed', type=int, default=2, help="Randomization seed")
parser.add_argument('--jitter', default=False, action="store_true", help="Jitter preset augmentation")
parser.add_argument('--scaling', default=False, action="store_true", help="Scaling preset augmentation")
parser.add_argument('--permutation', default=False, action="store_true", help="Equal Length Permutation preset augmentation")
parser.add_argument('--randompermutation', default=False, action="store_true", help="Random Length Permutation preset augmentation")
parser.add_argument('--magwarp', default=False, action="store_true", help="Magnitude warp preset augmentation")
parser.add_argument('--timewarp', default=False, action="store_true", help="Time warp preset augmentation")
parser.add_argument('--windowslice', default=False, action="store_true", help="Window slice preset augmentation")
parser.add_argument('--windowwarp', default=False, action="store_true", help="Window warp preset augmentation")
parser.add_argument('--rotation', default=False, action="store_true", help="Rotation preset augmentation")
parser.add_argument('--spawner', default=False, action="store_true", help="SPAWNER preset augmentation")
parser.add_argument('--dtwwarp', default=False, action="store_true", help="DTW warp preset augmentation")
parser.add_argument('--shapedtwwarp', default=False, action="store_true", help="Shape DTW warp preset augmentation")
parser.add_argument('--wdba', default=False, action="store_true", help="Weighted DBA preset augmentation")
parser.add_argument('--discdtw', default=False, action="store_true", help="Discrimitive DTW warp preset augmentation")
parser.add_argument('--discsdtw', default=False, action="store_true", help="Discrimitive shapeDTW warp preset augmentation")
parser.add_argument('--extra_tag', type=str, default="", help="Anything extra")

# TimeXer
parser.add_argument('--patch_len', type=int, default=16, help='patch length')

# DUET
parser.add_argument('--CI', action='store_true', help='channel independence', default=False)
parser.add_argument('--hidden_size', type=int, default=256, help='DUET hidden size')
parser.add_argument('--win_size', type=int, default=2, help='DUET window size')
parser.add_argument('--output_attention', default=False, action="store_true", help="output attention")
parser.add_argument('--stride', type=int, default=8, help='patch stride')
parser.add_argument('--period_len', type=int, default=4, help='period lenth')
parser.add_argument('--fc_dropout', type=float, default=0.2, help='fc dropout')
parser.add_argument('--num_experts', type=int, default=4, help='number of experts')
parser.add_argument('--noisy_gating', action='store_true', help='noisy gating', default=False)
parser.add_argument('--k', type=int, default=1, help='noisy gating top k')

args = parser.parse_args()

In [3]:
import re
from types import SimpleNamespace

args = SimpleNamespace(**vars(args))


In [4]:


# 假设 .sh 文件路径
sh_file_path = '/data/nishome/user1/chaochuan/TSGym_benchmark/scripts/long_term_forecast/NYSE_script/gym_non_Transformer/TSGym10000_False_False_RevIN_MoEMA_False_inverted-encoding_MLP_null_null_True_False_HP_96_64-256_2_30_DBLoss_0.0001_null.sh'

# 读取 .sh 文件内容
with open(sh_file_path, 'r') as file:
    lines = file.readlines()

# 正则表达式来匹配 --parameter value 形式的参数
for line in lines:
    # 清除空格并跳过空行或注释行
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    
    # 匹配 --parameter value 形式的参数
    match = re.match(r'--(\w+)\s+([^\s]+)', line)
    if match:
        param, value = match.groups()
        
        # 处理不同的参数类型
        # 如果值是数字或者浮点数，转换为数字类型
        if value.isdigit():
            value = int(value)
        elif re.match(r'^\d+\.\d+$', value):
            value = float(value)
        elif value.lower() in ['true', 'false']:
            value = value.lower() == 'true'  # 转换为布尔类型
        
        # 将解析出来的参数作为属性添加到 args 对象中
        setattr(args, param, value)

# 访问参数
print(args.task_name)  # 输出 'long_term_forecast'
print(args.seq_len)    # 输出 48
print(args.learning_rate)  # 输出 0.001


long_term_forecast
96
0.0001


In [5]:
args.task_name = 'finance_regressing'

In [6]:
args.data="finance"
args.root_path="/data/nishome/user1/chaochuan/TSGym_benchmark/dataset/finance_nyse"
args.data_path="finance_nyse.csv"

In [13]:
from data_provider.data_factory import data_provider

train_data, train_loader = data_provider(args, 'train')

Use GPU: cuda:0,1,2
train 771


In [14]:
for x,y,x_mark,y_mark in train_loader:
    print(x.shape, y.shape, x_mark.shape, y_mark.shape)
    break

torch.Size([32, 96, 5]) torch.Size([32, 1]) torch.Size([32, 96, 4]) torch.Size([32, 4])


In [15]:
x[0,-1]

tensor([-1.8016, -1.4547, -1.4690, -1.4223, -1.2771], device='cuda:0')

In [16]:
y[0]

tensor([0.8644], device='cuda:0')

In [7]:
device = torch.device("cuda" if args.use_gpu and torch.cuda.is_available() else "cpu")

In [8]:
for batch_x,batch_y,batch_x_mark,batch_y_mark in train_loader:
    dec_inp = torch.zeros_like(batch_y[:, -args.pred_len:, :]).float().to(device)
    dec_inp = torch.cat([batch_y[:, :args.label_len, :], dec_inp], dim=1).float().to(device)
    break

In [12]:
gym_x_mark_list = [True, False]
gym_series_sampling_list = [True, False]
gym_series_norm_list = ['None', 'Stat', 'RevIN', 'DishTS']
gym_series_decomp_list = ['None', 'MA', 'MoEMA', 'DFT']
gym_channel_independent_list = [False, True]
gym_input_embed_list = ['inverted-encoding', 'series-encoding', 'series-patching']
gym_network_architecture_list = ['Transformer', 'GRU'] # 
gym_attn_list =['self-attention', 'auto-correlation', 'sparse-attention', 'frequency-enhanced-attention', 'null', 'destationary-attention'] # 'destationary-attention',
gym_feature_attn_list = ['null', 'self-attention', 'sparse-attention']
gym_encoder_only_list = [True]
gym_frozen_list = [False, True]

In [13]:
def wrong_setting(series_sampling, series_norm, channel_independent, input_embed, network_architecture, attn, feature_attn, gym_frozen):
    if series_sampling and input_embed == 'inverted-encoding':
        return True
    if channel_independent and input_embed == 'inverted-encoding':
        return True
    if not channel_independent and input_embed == 'series-patching':
        return True
    if network_architecture == 'Transformer' and attn == 'null':
        return True
    if network_architecture in ['GRU','MLP'] and attn != 'null':
        return True
    if attn == 'destationary-attention' and series_norm != 'Stat':
        return True
    if attn == 'destationary-attention' and input_embed != 'series-encoding':
        return True
    if channel_independent and feature_attn != 'null':
        return True
    if gym_frozen and network_architecture not in ['LLM-GPT4TS', 'LLM-TimeLLM','TSFM-Timer', 'TSFM-Moment']:
        return True
    if network_architecture in ['LLM-GPT4TS', 'LLM-TimeLLM','TSFM-Timer', 'TSFM-Moment'] and attn != 'self-attention':
        return True
    if network_architecture == 'GRU' and input_embed == 'inverted-encoding':
        return True
    if input_embed == 'inverted-encoding' and attn not in ['self-attention', 'sparse-attention', 'null']:
        return True
    return False

In [14]:
from itertools import product
from tqdm import tqdm

# 使用 itertools.product 生成所有参数组合
for gym_x_mark, gym_series_sampling, gym_series_norm, gym_series_decomp, gym_channel_independent, \
    gym_input_embed, gym_network_architecture, gym_attn, gym_feature_attn, gym_encoder_only, gym_frozen \
        in tqdm(product(gym_x_mark_list, gym_series_sampling_list, gym_series_norm_list, gym_series_decomp_list,
                       gym_channel_independent_list, gym_input_embed_list, gym_network_architecture_list,
                       gym_attn_list, gym_feature_attn_list, gym_encoder_only_list, gym_frozen_list), 
                desc="Processing combinations", total=len(gym_x_mark_list) * len(gym_series_sampling_list) *
                     len(gym_series_norm_list) * len(gym_series_decomp_list) * len(gym_channel_independent_list) *
                     len(gym_input_embed_list) * len(gym_network_architecture_list) * len(gym_attn_list) *
                     len(gym_feature_attn_list) * len(gym_encoder_only_list) * len(gym_frozen_list)):
    if wrong_setting(gym_series_sampling, gym_series_norm, gym_channel_independent, gym_input_embed,gym_network_architecture, gym_attn, gym_feature_attn, gym_frozen):
        continue # 冲突setting，跳过
    else:
        try:
            model = TSGym(args,gym_x_mark=gym_x_mark,gym_series_sampling=gym_series_sampling,gym_series_norm=gym_series_norm,gym_series_decomp=gym_series_decomp,gym_channel_independent=gym_channel_independent,gym_input_embed=gym_input_embed,gym_network_architecture=gym_network_architecture,gym_attn=gym_attn,gym_feature_attn=gym_feature_attn,gym_encoder_only=gym_encoder_only,gym_frozen=gym_frozen).float().to(device)
            preds = model(batch_x, batch_x_mark, dec_inp, batch_y_mark)
        except Exception as e:
            print(f"Error with configuration: x_mark={gym_x_mark}, series_sampling={gym_series_sampling}, series_norm={gym_series_norm}, series_decomp={gym_series_decomp}, channel_independent={gym_channel_independent}, input_embed={gym_input_embed}, network_architecture={gym_network_architecture}, attn={gym_attn}, feature_attn={gym_feature_attn}, encoder_only={gym_encoder_only}, frozen={gym_frozen}: {e}")


Processing combinations:   0%|          | 0/27648 [00:00<?, ?it/s]

Processing combinations: 100%|██████████| 27648/27648 [05:30<00:00, 83.65it/s] 


In [ ]:
Error with configuration: x_mark=True, series_sampling=True, series_norm=None, series_decomp=None, channel_independent=False, input_embed=series-encoding, network_architecture=Transformer, attn=destationary-attention, feature_attn=null, encoder_only=True, frozen=False: 
